In [46]:
import pandas as pd
import numpy as np
import pyarrow as pa

df = pd.read_csv("../data/cars.csv")
print("Start:", df.shape)

Start: (762091, 20)


In [47]:
df.describe()

,year,mileage,accidents_or_damage,one_owner,personal_use_only,seller_rating,driver_rating,driver_reviews_num,price_drop,price
count,762091.000000,7.615850e+05,737879.000000,730608.000000,737239.000000,548118.000000,730459.000000,762091.000000,410112.000000,7.620910e+05
mean,2017.791398,5.578169e+04,0.228616,0.561969,0.657212,4.158568,4.623523,89.550900,1007.467068,3.648898e+04
std,5.110532,4.355788e+04,0.419942,0.496145,0.474642,0.805741,0.276902,115.082266,1375.122208,1.984183e+06
min,1915.000000,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,100.000000,1.000000e+00
25%,2016.000000,2.328700e+04,0.000000,0.000000,0.000000,3.800000,4.500000,14.000000,380.000000,1.958300e+04
50%,2019.000000,4.559600e+04,0.000000,1.000000,1.000000,4.500000,4.700000,51.000000,642.000000,2.798900e+04
75%,2021.000000,7.836500e+04,0.000000,1.000000,1.000000,4.700000,4.800000,119.000000,1007.000000,3.948800e+04
max,2024.000000,1.119067e+06,1.000000,1.000000,1.000000,5.000000,5.000000,1025.000000,170995.000000,1.000000e+09


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 762091 entries, 0 to 762090
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   manufacturer         762091 non-null  str    
 1   model                762091 non-null  str    
 2   year                 762091 non-null  int64  
 3   mileage              761585 non-null  float64
 4   engine               747041 non-null  str    
 5   transmission         752187 non-null  str    
 6   drivetrain           740529 non-null  str    
 7   fuel_type            739164 non-null  str    
 8   mpg                  620020 non-null  str    
 9   exterior_color       753232 non-null  str    
 10  interior_color       705116 non-null  str    
 11  accidents_or_damage  737879 non-null  float64
 12  one_owner            730608 non-null  float64
 13  personal_use_only    737239 non-null  float64
 14  seller_name          753498 non-null  str    
 15  seller_rating        548118 

In [49]:
#dropping the missing mileage rows
df = df.dropna(subset=["mileage"])
print(df.shape)

(761585, 20)


#### Dealing with price only including values between 2k - 200k

In [50]:
#dropping rows for price outside of range 2k - 200k
df=df[df["price"].between(2000, 200000)]

In [51]:
df["had_price_drop"] = (df["price_drop"] > 0).astype(int)
df["price_drop"] = df["price_drop"].fillna(0)

#### Dealing with seller rating, adding flag variable

In [52]:
g = df.groupby("seller_name")["seller_rating"]
mixed = g.apply(lambda s: s.isna().any() and s.notna().any())
print("Sellers with some ratings missing and some present:", mixed.sum())

Sellers with some ratings missing and some present: 337


In [53]:
df["seller_rating"] = df["seller_rating"].fillna(
    df.groupby("seller_name")["seller_rating"].transform("median")
)

In [54]:
df["seller_rating_missing"] = df["seller_rating"].isna().astype(int)

#### Dealing with MPG grouping cars that have same make/model/year and giving them the same mpg

In [55]:
df[["year","manufacturer", "model","mpg"]]

,year,manufacturer,model,mpg
0,2013,Acura,ILX Hybrid 1.5L,39-38
1,2013,Acura,ILX Hybrid 1.5L,39-38
2,2013,Acura,ILX Hybrid 1.5L,39-38
3,2013,Acura,ILX Hybrid 1.5L,39-38
4,2013,Acura,ILX Hybrid 1.5L,39-38
...,...,...,...,...
762086,2022,Volvo,S60 B5 Momentum,25-33
762087,2012,Volvo,S60 T5,NaN
762088,2014,Volvo,S60 T5,21-30
762089,2013,Volvo,S60 T5 Platinum,20-29


In [56]:
mpg_split = df["mpg"].str.split("-", expand=True)
df["mpg_city"] = pd.to_numeric(mpg_split[0], errors="coerce")
df["mpg_hwy"] = pd.to_numeric(mpg_split[1], errors = 'coerce')

In [57]:
df[["mpg_city","mpg_hwy"]].describe()

,mpg_city,mpg_hwy
count,618728.000000,615877.000000
mean,21.316918,27.953605
std,6.500205,6.937287
min,0.000000,0.000000
25%,17.000000,24.000000
50%,20.000000,28.000000
75%,24.000000,32.000000
max,126.000000,255.000000


In [58]:
for col in ["mpg_city", "mpg_hwy"]:
    before =df[col].isna().sum()

    df[col] = df[col].fillna(df.groupby(["manufacturer","year","model"])[col].transform("median"))

    df[col] = df[col].fillna(df.groupby(["manufacturer", "model"])[col].transform("median"))

    print(f"{col}: filled {before - df[col].isna().sum():,}, still missing {df[col].isna().sum():,}")

mpg_city: filled 109,809, still missing 31,443
mpg_hwy: filled 112,400, still missing 31,703


In [59]:
for c in ["mpg_city", "mpg_hwy"]:
    df.loc[~df[c].between(5, 150), c] = np.nan

#### Fuel Type

In [60]:
df["fuel_type"].value_counts()

fuel_type
Gasoline                         642774
Hybrid                            29106
Diesel                            27900
E85 Flex Fuel                     18748
Electric                          16174
B                                  1442
Flexible Fuel                       490
Plug-In Hybrid                      118
Gasoline Fuel                        80
Gasoline/Mild Electric Hybrid        70
Regular Unleaded                     56
Premium Unleaded                     49
G                                    45
Compressed Natural Gas               29
Unspecified                          26
Diesel Fuel                           5
Hybrid Fuel                           5
Other                                 3
Gaseous                               3
Hydrogen Fuel Cell                    3
Flex Fuel Capability                  2
Gas                                   2
PHEV                                  2
Electric Fuel System                  2
Gas/Electric Hybrid           

In [61]:
fuel_groups = {
    "Gasoline": ["Gasoline", "Gasoline Fuel", "Regular Unleaded", "Premium Unleaded",
                 "Premium", "Premium (Required)", "Gas", "G"],
    "Flex Fuel": ["E85 Flex Fuel", "Flexible Fuel", "Flex Fuel Capability",
                  "Flex Fuel", "E85 Fl", "Bi-Fuel"],
    "Diesel": ["Diesel", "Diesel Fuel", "Biodiesel", "Bio Diesel"],
    "Hybrid": ["Hybrid", "Hybrid Fuel", "Gasoline/Mild Electric Hybrid",
               "Gas/Electric Hybrid", "B"],
    "Plug-In Hybrid": ["Plug-In Hybrid", "PHEV", "Plug-In Electric/Gas"],
    "Electric": ["Electric", "Electric Fuel System"],
    "Other": ["Compressed Natural Gas", "Natural Gas", "Gaseous",
              "Hydrogen Fuel Cell", "Other"],
}

fuel_map = {variant: clean for clean, variants in fuel_groups.items() for variant in variants}

df["fuel_type"] = df["fuel_type"].map(fuel_map).fillna("Unknown")
df["fuel_type"].value_counts()

fuel_type
Gasoline          643008
Hybrid             30625
Diesel             27907
Unknown            22861
Flex Fuel          19243
Electric           16176
Plug-In Hybrid       121
Other                 39
Name: count, dtype: int64

#### Drivetrain

In [62]:
df["drivetrain"].value_counts()

drivetrain
Front-wheel Drive                                              241192
All-wheel Drive                                                229278
Four-wheel Drive                                               156916
Rear-wheel Drive                                                96837
FWD                                                              6444
AWD                                                              3574
4WD                                                              1921
RWD                                                              1749
All-Wheel Drive                                                   120
Unknown                                                           104
Front-Wheel Drive                                                 104
Front-Wheel Drive with Limited-Slip Differential                   44
Four-Wheel Drive                                                   43
Four-Wheel Drive with Locking and Limited-Slip Differential        42
Rear-Whee

mapping drivetrains to 4WD, AWD, FWD, and RWD, and going to fill in unknowns with most common drivetrains for make/model

In [63]:
d = df["drivetrain"].fillna("").str.lower()

conditions = [
    d.str.contains("four|4wd|4x4"),
    d.str.contains("all|awd"),
    d.str.contains("front|fwd"),
    d.str.contains("rear|rwd"),
]
choices = ["4WD", "AWD", "FWD", "RWD"]

df["drivetrain"] = np.select(conditions, choices, default = "Unknown")
df["drivetrain"].value_counts()

drivetrain
FWD        247801
AWD        233036
4WD        158958
RWD         98640
Unknown     21545
Name: count, dtype: int64

In [64]:
def confident_mode(s, threshold=0.9):
    s = s[s != "Unknown"]
    if s.empty:
        return np.nan
    shares = s.value_counts(normalize=True)
    return shares.index[0] if shares.iloc[0] >= threshold else np.nan

group_mode = df.groupby(["manufacturer", "model", "year"])["drivetrain"].transform(confident_mode)

unknown = df["drivetrain"] == "Unknown"
before = unknown.sum()
df.loc[unknown, "drivetrain"] = group_mode[unknown].fillna("Unknown")
print(f"Filled {before - (df['drivetrain'] == 'Unknown').sum():,} of {before:,} unknowns")

Filled 8,077 of 21,545 unknowns


#### Engine

In [65]:
df["engine"].value_counts()

engine
2.0L I4 16V GDI DOHC Turbo                                          75406
3.6L V6 24V MPFI DOHC                                               35412
3.6L V6 24V GDI DOHC                                                27010
2.0L I4 16V MPFI DOHC                                               19522
1.5L I4 16V GDI DOHC Turbo                                          18171
                                                                    ...  
2.4L I-5 DOHC, variable valve control, intercooled turbo, engine        1
2.5L DOHC turbocharged I5 engine w/intercooler                          1
2.0L Turbo I4 250hp 258ft                                               1
300.0HP 2.5L 5 Cylinder Engine Gasoline Fuel                            1
2.5L 5-cylinder turbocharged engine                                     1
Name: count, Length: 6849, dtype: int64

Since there are so many different types of engine types with different categories, I am going to remove it for now to get a working model then will decide on the categories to use later


In [66]:
df= df.drop(columns=["engine"])

#### Transmission

In [67]:
df["transmission"].value_counts()

transmission
6-Speed Automatic                               148517
8-Speed Automatic                               139483
Automatic CVT                                   108559
Automatic                                        96724
9-Speed Automatic                                60913
                                                 ...  
Automsmatic                                          1
1-SPEED G                                            1
Automatic, 8-Spd Geartronic w/Adaptive Shift         1
8-SPEED A                                            1
9Speed AT                                            1
Name: count, Length: 1309, dtype: int64

In [68]:
t = df["transmission"].fillna("").str.lower()

conditions = [
    t.str.contains(r"cvt|variable"),
    t.str.contains(r"manual|m/t|\bmt\b|stick"),
    t.str.contains(r"auto|a/t|\bat\b|\ba\b|tiptronic|geartronic|dual.clutch|dct|pdk"),
]
choices = ["CVT", "Manual", "Automatic"]
df["transmission_type"] = np.select(conditions, choices, default="Unknown")

df["transmission_type"].value_counts()

transmission_type
Automatic    604106
CVT          117668
Manual        23766
Unknown       14440
Name: count, dtype: int64

In [69]:
df.loc[df["transmission_type"] == "Unknown", "transmission"].value_counts().head(20)

transmission
IVT                               1060
6-Speed                            893
Transmission w/Dual Shift Mode     809
8-Speed                            408
5-Speed                            216
7-Speed                            191
SHIFTRONIC                         131
NOT SPECIFIED                       92
SINGLE-SPEED TRANSMISSION           90
AWD                                 77
Not Specified                       76
10-SPEED TRANSMISSION               75
OTHER                               50
FWD                                 28
SELECT                              25
Single-Speed Fixed Gear             25
Single Speed Reducer                20
Select                              18
UNKNOWN                             17
M                                   16
Name: count, dtype: int64

In [70]:
df = df.drop(columns=["transmission"])

#### Driver Rating

In [71]:
missing = df["driver_rating"].isna()
df.loc[missing, "driver_reviews_num"].value_counts().head()

driver_reviews_num
0.0    30926
Name: count, dtype: int64

All 30,926 missing driver_rating values have driver_reviews_num = 0, meaning the model has no owner reviews. Left as NaN instead of filling, since 'unrated' isn't the same as 'average'.

#### Accidents or Damage, One Owner, and Personal Use Only

In [72]:
df["one_owner"].value_counts(dropna=False)

one_owner
1.0    409593
0.0    319188
NaN     31199
Name: count, dtype: int64

In [73]:
df["personal_use_only"].value_counts(dropna=False)

personal_use_only
1.0    483212
0.0    252046
NaN     24722
Name: count, dtype: int64

In [74]:
for col in ["accidents_or_damage", "one_owner", "personal_use_only"]:
    df[col] = (
        df[col]
        .map({1: "Yes", 0: "No"})
        .fillna("Unknown")
        .astype("category")
    )

#### Interior/Exterior Colour

I am going to drop the colour columns for now since they don't have too much of an effect on the price, and there are so many different types of colours by make and model. Eventually I will map the colours a general colour ie "Ivory" to something like just "white"

In [75]:
df = df.drop(columns = ["interior_color"])
df = df.drop(columns = ["exterior_color"])

#### Mileage

In [76]:
df = df.dropna(subset=["mileage"])

In [77]:
df["mileage"].describe()

count    7.599800e+05
mean     5.584172e+04
std      4.348415e+04
min      0.000000e+00
25%      2.339400e+04
50%      4.567200e+04
75%      7.841400e+04
max      1.119067e+06
Name: mileage, dtype: float64

In [78]:
df.nlargest(10, "mileage")[["manufacturer", "model", "year", "mileage", "price"]]

,manufacturer,model,year,mileage,price
756847,Volvo,XC70 3.2L,2010,1119067.0,16997.0
494409,Lincoln,Continental,1967,999999.0,22900.0
511961,Mercedes-Benz,GLK-Class GLK 350,2011,999999.0,13995.0
617503,Porsche,356 A,1959,974302.0,179500.0
319941,Honda,Civic LX,2013,938032.0,14816.0
522994,Mercedes-Benz,E-Class E 350 4MATIC,2021,915383.0,53539.0
136180,Chevrolet,Silverado 3500 LTZ,2013,777698.0,16999.0
669643,Toyota,Camry LE,2020,769938.0,21825.0
229883,Ford,F-350 Lariat Super Duty,2014,763474.0,23995.0
302334,Honda,Pilot LX,2020,754680.0,20499.0


In [79]:
df = df[df["mileage"].notna() & (df["mileage"] <= 400000)]

In [80]:
df["mileage"].describe()

count    759917.000000
mean      55800.680764
std       43221.470088
min           0.000000
25%       23392.000000
50%       45668.000000
75%       78400.000000
max      398677.000000
Name: mileage, dtype: float64

In [81]:
error_mileage = (df["mileage"] < 100) & (df["year"] <= 2021)
df = df[~error_mileage]

Deleted missing mileage on cars, removed any car with mileage over 400k, and removed cars with mileage less than 100 and the year less than 2021

#### Checking Duplicates, impossible years, and removing leftover columns

In [82]:
df.duplicated().sum()

np.int64(9126)

In [83]:
df = df.drop_duplicates()

In [84]:
key = ["manufacturer", "model", "year", "mileage", "price", "seller_name"]
df = df.drop_duplicates(subset=key)

In [85]:
df["year"].describe()

count    748817.000000
mean       2017.833155
std           4.907181
min        1919.000000
25%        2016.000000
50%        2019.000000
75%        2021.000000
max        2024.000000
Name: year, dtype: float64

In [86]:
df.info()

<class 'pandas.DataFrame'>
Index: 748817 entries, 0 to 762090
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   manufacturer           748817 non-null  str     
 1   model                  748817 non-null  str     
 2   year                   748817 non-null  int64   
 3   mileage                748817 non-null  float64 
 4   drivetrain             748817 non-null  str     
 5   fuel_type              748817 non-null  str     
 6   mpg                    610417 non-null  str     
 7   accidents_or_damage    748817 non-null  category
 8   one_owner              748817 non-null  category
 9   personal_use_only      748817 non-null  category
 10  seller_name            740392 non-null  str     
 11  seller_rating          546790 non-null  float64 
 12  driver_rating          718590 non-null  float64 
 13  driver_reviews_num     748817 non-null  float64 
 14  price_drop             748817 non-nu

In [87]:
df = df.drop(columns=["mpg"])

In [88]:
cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type"]
df[cat_cols] = df[cat_cols].astype("category")

In [89]:
df.to_parquet("../data/cars_clean.parquet", index = False)